# RAG 고도화 및 최적화 — 실습

> **먼저 [`rag_pdf.ipynb`](./rag_pdf.ipynb)(기본 RAG 실습)을 끝내고 오자.** 환경 세팅·커널이 거기에 있고, 같은 커널 **RAG PDF Practice** 를 그대로 쓴다.

블로그 글 [RAG 고도화 및 최적화](https://seojuny.dev/advanced-rag) 의 실습이다.

방식은 **누적**이다. 가장 단순한 **기본 RAG(STAGE 0)** 에서 시작해, 개선을 **하나씩 적용**하고 **직전 단계와 답을 비교**한다. 그 개선이 무엇을 고치는지 눈으로 확인한 뒤 그 위에 다음 개선을 쌓는다. 단계를 다 쌓으면 그게 **고도화 RAG**다.

```
STAGE 0 기본 → +표보존 → +구조청킹 → +Contextual → +쿼리재작성 → +하이브리드 → +리랭킹 → +출처인용 = 고도화
```

각 단계는 **그 개선의 효과가 드러나는 질문**으로 비교한다. 개선이 답을 바꾸면 답으로, 검색만 바꾸면 순위·점수로 보여 준다. 셀의 `question` 을 직접 바꿔 가며 실험해 보면 좋다 — 효과가 잘 보이는 추천 질문을 주석으로 달아 뒀다.

데이터는 가상 회사 **노바테크의 사내 문서 PDF 7개**(`../data_advanced/`): 취업규칙·복리후생·급여규정·보안정책·인사규정·출장여비규정·사내FAQ.

> 임베딩·LLM 호출에는 약간의 무작위성이 있어 매 실행이 100% 똑같진 않다.

## 준비 — LLM 클라이언트
`get_client()` : `OPENAI_API_KEY` 가 있으면 OpenAI, 없으면 로컬 Ollama. (답변·임베딩 모델을 함께 돌려준다)

In [ ]:
import os
import re
import math
from collections import Counter
from pathlib import Path

from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()  # 상위 폴더의 .env 에서 OPENAI_API_KEY 를 읽는다


def get_client():
    """OPENAI_API_KEY 가 있으면 OpenAI, 없으면 로컬 Ollama 를 쓴다. (client, 답변 모델, 임베딩 모델)"""
    if os.getenv("OPENAI_API_KEY", "").strip().startswith("sk-"):
        return OpenAI(), "gpt-4o-mini", "text-embedding-3-small"
    return OpenAI(base_url="http://localhost:11434/v1", api_key="ollama"), "qwen2.5", "nomic-embed-text"


client, CHAT_MODEL, EMBEDDING_MODEL = get_client()
print("답변:", CHAT_MODEL, "| 임베딩:", EMBEDDING_MODEL)

## 라이브러리 설치 (한 번)

고도화 실습에 쓰는 라이브러리를 **호환되는 버전으로 한 번에** 설치한다. 특히 `langchain` 은 1.x 와 0.3 의 구조가 크게 달라, retriever 컴포넌트·`ragas` 와 맞물리는 **0.3 으로 고정**한다. (이미 깔려 있으면 건너뛴다.)

In [ ]:
import sys, subprocess
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
    "rank-bm25", "kiwipiepy", "sentence-transformers",
    # langchain 은 0.3 라인으로 고정 — 1.x 는 langchain-core 1.x 를 요구해 retriever·ragas 와 충돌한다.
    # langchain-core<1.0 을 핀으로 잡으면 langchain-openai 등 생태계가 모두 0.3 호환으로 풀린다.
    "langchain-core<1.0", "langchain<1.0", "langchain-community<0.4",
    "langchain-openai<1.0", "langchain-chroma<1.0", "langchain-text-splitters<1.0",
    "langchain-google-vertexai", "ragas==0.2.15"])
print("라이브러리 준비 완료")

## 공통 함수 — 임베딩·인덱스·검색·답변·비교

단계마다 새로 만드는 건 '인덱스(무엇을 저장하나)'와 '검색(어떻게 찾나)'뿐이다. 임베딩·답변·비교 함수는 처음에 한 번 정의해 계속 쓴다.
메모리 벡터 DB(`EphemeralClient`)라 단계별 인덱스가 서로 격리된다(디스크에 안 남는다).

In [ ]:
import chromadb


def embed(texts):
    """텍스트(또는 텍스트 리스트)를 임베딩 벡터로 변환한다."""
    if isinstance(texts, str):
        texts = [texts]
    response = client.embeddings.create(model=EMBEDDING_MODEL, input=texts)
    return [item.embedding for item in response.data]


def make_index(items, name):
    """items(=[{"text","source"}, ...])를 임베딩해 검색 가능한 메모리 컬렉션으로 만든다."""
    items = [it for it in items if it["text"].strip()]
    embeddings = embed([it["text"] for it in items])

    # EphemeralClient 는 인메모리 백엔드를 공유하므로, 같은 이름이 있으면 지우고 새로 만든다.
    # (이러지 않으면 셀을 다시 실행할 때 이전 청크가 남아 검색이 꼬인다)
    db = chromadb.EphemeralClient()
    try:
        db.delete_collection(name)
    except Exception:
        pass
    collection = db.get_or_create_collection(name, metadata={"hnsw:space": "cosine"})
    collection.add(
        ids=[f"chunk-{i}" for i in range(len(items))],
        embeddings=embeddings,
        documents=[it["text"] for it in items],
        metadatas=[{"source": it["source"]} for it in items],
    )
    return collection


def search(collection, query, k=3, where=None):
    """질문과 의미가 가까운 청크 k개를 찾는다 (where 로 특정 문서만 좁힐 수 있다)."""
    result = collection.query(
        query_embeddings=[embed(query)[0]],
        n_results=k,
        where=where,
        include=["documents", "metadatas"],
    )
    documents = result["documents"][0]
    metadatas = result["metadatas"][0]
    return [{"text": text, "source": meta["source"]} for text, meta in zip(documents, metadatas)]


def answer_plain(question, chunks):
    """청크를 근거로 평이하게 답한다 (출처 표기 없음)."""
    context = "\n\n".join(f"({c['source']}) {c['text']}" for c in chunks)
    response = client.chat.completions.create(
        model=CHAT_MODEL,
        temperature=0,
        messages=[
            {"role": "system", "content": "사내 문서를 근거로만 답하고, 문서에 없으면 모른다고 답하라."},
            {"role": "user", "content": f"[문서]\n{context}\n\n[질문]\n{question}"},
        ],
    )
    return response.choices[0].message.content


def compare_stage(question, before, after, before_label, after_label):
    """같은 질문에 '개선 전'과 '개선 후' 파이프라인의 답을 나란히 출력한다.
    before/after 는 질문을 받아 답을 돌려주는 함수다."""
    print("Q.", question)
    print(f"\n[{before_label}]\n" + before(question))
    print(f"\n[{after_label}]\n" + after(question))

## STAGE 0 — 기본 RAG (출발점)

`rag_pdf.ipynb` 와 같은 수준의 단순 RAG다: **pypdf 단순 추출 · 글자수 청킹 · dense 검색 · 평이한 답변.**
여기서부터 개선을 하나씩 쌓는다. 표 속 값을 질의하면 기본 RAG는 찾지 못한다.

In [ ]:
import pypdf


def pypdf_text(path):
    """pypdf 로 페이지 텍스트를 이어 붙인다 (표 구조는 깨진다)."""
    pages = pypdf.PdfReader(path).pages
    return "\n".join(page.extract_text() or "" for page in pages)


def char_chunk(text, size=500, overlap=50):
    """글자 수로 단순하게 자른다. overlap 만큼 겹쳐 경계 문맥을 잇는다."""
    chunks = []
    start = 0
    while start < len(text):
        chunks.append(text[start:start + size])
        start += size - overlap
    return chunks


PDF_PATHS = sorted(Path("../data_advanced").glob("*.pdf"))

# 인덱스 v0: pypdf 단순 추출 + 글자수 청킹
items_v0 = [
    {"text": chunk, "source": pdf.stem}
    for pdf in PDF_PATHS
    for chunk in char_chunk(pypdf_text(str(pdf)))
]
col_v0 = make_index(items_v0, "stage0")


def rag_v0(question):
    """STAGE 0 기본 RAG."""
    return answer_plain(question, search(col_v0, question, k=3))


# question 을 바꿔 직접 실험해 보자.
question = "부장의 연봉 환산 금액은 얼마야?"
print(rag_v0(question))

## STAGE 1 — +표보존 추출

**문제.** 기본 RAG는 `pypdf` 로 본문·표를 가리지 않고 텍스트만 뽑는다. 그러면 직급별 표가 `사원 3,000,000 대리 3,600,000 …` 처럼 행·열 없이 한 줄로 뒤섞이고, 표가 길수록 끝 행 값이 검색에서 누락되기 쉽다.

**개선.** `pymupdf4llm` 같은 레이아웃 파서는 표를 **행·열이 살아 있는 Markdown 표**로 바꾼다. 효과는 추출된 텍스트에서 바로 보인다.

> 단, 답은 아직 안 바뀐다 — 글자수 청킹이 표를 다시 쪼개기 때문. STAGE 2에서 고친다.

In [ ]:
import pymupdf4llm

# 표를 살린 Markdown 으로 PDF 를 읽어 둔다 (문서별 원문)
RAW = {pdf.stem: pymupdf4llm.to_markdown(str(pdf), show_progress=False) for pdf in PDF_PATHS}

flat = pypdf_text("../data_advanced/급여규정.pdf")
markdown = RAW["급여규정"]

print("─ pypdf (단순) ─ 표가 줄줄이 늘어진다:")
i = flat.find("직급")
print(flat[i:i + 95])

print("\n─ pymupdf4llm (레이아웃) ─ 행·열이 보존된 Markdown 표:")
j = markdown.find("|직급")
print(markdown[j:j + 120])

# 인덱스 v1: pymupdf 표보존 추출 + (아직) 글자수 청킹
items_v1 = [{"text": chunk, "source": source} for source, text in RAW.items() for chunk in char_chunk(text)]
col_v1 = make_index(items_v1, "stage1")


def rag_v1(question):
    """STAGE 1: 표보존 추출 + 글자수 청킹."""
    return answer_plain(question, search(col_v1, question, k=3))

## STAGE 2 — +구조 청킹

**문제.** 글자 수로 자르면 Markdown 표가 청크 경계에서 또 쪼개진다.

**개선.** 규정 문서는 `## 헤딩` 이나 `제N조` 단위로 내용이 묶인다. 그 경계부터 우선해 자르면 표·항목이 한 청크에 통째로 담긴다. 표보존(STAGE 1)과 합쳐져 이제 표 속 값을 묻는 질문에 답이 나온다.

비교: 직전 v1 → 이번 v2.

In [ ]:
def struct_chunk(text, max_size=700):
    """구조 경계(## 헤딩 / 제N조)에서 자른다. 너무 길면 글자수로 보조 분할."""
    chunks = []
    for part in re.split(r"(?=^#+ )|(?=제\d+조)", text, flags=re.M):
        part = part.strip()
        if not part:
            continue
        if len(part) <= max_size:
            chunks.append(part)
        else:
            chunks.extend(char_chunk(part, max_size, overlap=60))
    return chunks


# 인덱스 v2: 표보존 추출 + 구조 청킹
items_v2 = [{"text": chunk, "source": source} for source, text in RAW.items() for chunk in struct_chunk(text)]
col_v2 = make_index(items_v2, "stage2")


def rag_v2(question):
    """STAGE 2: 표보존 + 구조 청킹."""
    return answer_plain(question, search(col_v2, question, k=3))


# question 을 바꿔 직접 실험해 보자. 표 속 값을 묻는 추천 질문:
#   - "차장의 연차 가산일은 며칠이야?"   (인접한 두 표를 헷갈리기 쉽다)
#   - "과장 월 기본급은 얼마야?"
question = "부장의 연봉 환산 금액은 얼마야?"
compare_stage(question, rag_v1, rag_v2, "v1 글자수 청킹", "v2 +구조 청킹")

## STAGE 3 — +Contextual Retrieval

**문제.** 검색은 청크 단위라, 청크를 문서에서 떼면 맥락이 사라진다. "사유 발생일로부터 **30일 이내**에 신청한다" 같은 청크엔 정작 '경조사비'라는 말이 없어(상위 헤딩에만 있다), 막연히 물으면 검색에서 밀린다.

**개선.** [Contextual Retrieval](https://www.anthropic.com/news/contextual-retrieval)은 임베딩 전에 **'어느 문서의 무엇'인지 한 문장**을 LLM 으로 만들어 청크 앞에 붙인다.

비교: 직전 v2 → 이번 v3.

In [ ]:
def add_context(chunk, source, full_doc):
    """청크 앞에 '어느 문서의 무엇'인지 한 문장을 LLM 으로 만들어 붙인다 (Contextual Retrieval)."""
    response = client.chat.completions.create(
        model=CHAT_MODEL,
        temperature=0,
        messages=[{"role": "user", "content":
            f"<문서 name='{source}'>{full_doc}</문서>\n"
            f"위 문서에서 아래 청크가 무엇에 대한 내용인지 한 문장으로.\n청크:{chunk}"}],
    )
    context_line = response.choices[0].message.content.strip()
    return context_line + "\n" + chunk


# 인덱스 v3: 구조 청킹 + 청크마다 맥락 한 문장 (인덱싱 때 청크 수만큼 LLM 호출)
items_v3 = []
for source, full_doc in RAW.items():
    for chunk in struct_chunk(full_doc):
        if chunk.strip():
            items_v3.append({"text": add_context(chunk, source, full_doc), "source": source})
col_v3 = make_index(items_v3, "stage3")


# Contextual 로 붙인 맥락이 실제로 어떻게 들어갔는지 몇 개 뽑아 본다.
print("─ Contextual: '붙인 맥락 문장' + '원래 청크' 예시 ─")
shown = 0
for item in items_v3:
    context_line, _, original = item["text"].partition("\n")
    if len(original) < 120:            # 너무 짧은 청크는 건너뛴다 (차이가 잘 안 보임)
        continue
    print("\n" + "=" * 70)
    print(f"[출처] {item['source']}")
    print(f"[붙인 맥락] {context_line}")
    print(f"[원래 청크] {original[:200].strip()} …")
    shown += 1
    if shown == 3:
        break
print("=" * 70)


def rag_v3(question):
    """STAGE 3: + Contextual."""
    return answer_plain(question, search(col_v3, question, k=3))


# 청크에 주제어가 빠져 있어 막연히 묻는 추천 질문:
#   - "사유 발생일로부터 며칠 이내에 신청해?"
#   - "90일은 무슨 기간이야?"
question = "30일 이내에 신청해야 하는 게 뭐야?"
compare_stage(question, rag_v2, rag_v3, "v2 맥락 없음", "v3 +Contextual")

## STAGE 4 — +쿼리 재작성

여기서부터는 인덱스(v3)는 그대로 두고 **검색 전에 질문을 다듬는다.**

**문제.** 직원 질문이 늘 검색하기 좋은 형태는 아니다 — 구어체("노트북 집에 가져가도 돼?")거나 문서 표현과 어긋난다("야근" ↔ "연장근로"). 그대로 임베딩하면 관련 없는 문서가 검색된다.

**개선.** 쿼리 변환의 가장 기본은 **쿼리 재작성**이다. LLM 에게 질문을 검색 친화적인 짧은 검색어로 다시 쓰게 한다. "노트북 집에 가져가도 돼?"는 "노트북 반출 규정"으로 바뀌고, 검색이 복리후생·FAQ 대신 **보안정책을 가져온다.**

비교: 직전 v3 → 이번 v4.

In [ ]:
def rewrite(question):
    """질문을 검색하기 좋은 짧은 검색어로 다시 쓴다."""
    response = client.chat.completions.create(
        model=CHAT_MODEL,
        temperature=0,
        messages=[{"role": "user", "content":
            f"다음 질문을 사내 규정 문서에서 검색하기 좋게, 핵심 명사 위주의 짧은 검색어로 다시 써라. "
            f"설명 없이 검색어만.\n질문:{question}"}],
    )
    return response.choices[0].message.content.strip()


def retrieve_rewrite(question, k=4):
    """질문을 재작성한 뒤 검색한다."""
    return search(col_v3, rewrite(question), k=k)


def rag_v4(question):
    """STAGE 4: + 쿼리 재작성."""
    return answer_plain(question, retrieve_rewrite(question))


# question 을 바꿔 실험해 보자. 구어체·표현이 어긋난 추천 질문 (재작성이 검색을 맞는 문서로 돌린다):
#   - "야근하면 돈 더 줘?"   → "연장근로수당"
#   - "월급 언제 들어와?"    → "급여 지급일"
question = "노트북 집에 가져가도 돼?"
rewritten = rewrite(question)
v3_hits = search(col_v3, question, k=3)     # 원 질문 그대로 검색
v4_hits = search(col_v3, rewritten, k=4)    # 재작성한 질문으로 검색 (출처·답변에 같은 결과를 쓴다)

print("원 질문:", question)
print("재작성:", rewritten)
print("\nv3 원 질문 검색 출처:", [h["source"] for h in v3_hits])
print("v4 재작성 검색 출처:", [h["source"] for h in v4_hits])
print("\n[v3 원 질문 그대로]\n" + answer_plain(question, v3_hits))
print("\n[v4 +쿼리 재작성]\n" + answer_plain(question, v4_hits))

### 더해보기 — 쿼리 분해 (직접 구현)

재작성이 '한 질문을 더 잘' 만드는 거라면, **쿼리 분해**는 '여러 정보를 따로' 찾는 것이다. "A는 얼마고 B는 며칠이야?"처럼 두세 문서를 합쳐야 하는 질문은, 한 번 검색하면 한쪽에 쏠려 다른 쪽을 놓친다. 질문을 정보별 검색어로 **쪼개** 각각 검색하면 빠짐없이 가져온다.

아래 `decompose()` 는 **뼈대만** 있다. 의사코드 주석을 보고 직접 채워, STAGE 4 재작성과 비교해 보자.

In [ ]:
def decompose(question):
    """[더해보기] 복합 질문을 정보별 검색어 2~3개로 쪼갠다 (쿼리 분해).
    아래 의사코드를 채워 직접 구현해 보자."""
    # 1. LLM 에게 질문을 핵심 검색어 2~3개로 나눠 달라고 요청한다 (temperature=0)
    #    프롬프트 예: "다음 질문에 답하려면 무엇을 찾아야 하는지 짧은 검색어 2~3개로 나눠라. 각 줄에 하나씩."
    # 2. 응답을 줄 단위(splitlines)로 쪼개고, 각 줄 앞의 번호·기호(1. - • 등)를 제거한다
    #    힌트: re.sub(r"^[\s\d.)\-•]+", "", line).strip()
    # 3. 빈 줄을 거르고 검색어 리스트를 반환한다
    raise NotImplementedError("decompose() 를 직접 구현해 보자")


# 구현했다면 아래 retrieve_decompose 의 주석을 풀어 STAGE 4 재작성과 비교해 보자:
# def retrieve_decompose(question):
#     hits, seen = [], set()
#     for sub_query in decompose(question):       # 쪼갠 검색어마다 따로 검색
#         for hit in search(col_v3, sub_query, k=2):
#             if hit["text"] not in seen:
#                 seen.add(hit["text"]); hits.append(hit)
#     return hits
#
# 추천 질문 (세 문서에 흩어진 정보 — 효과가 크게 보인다):
#   "대리 연차 가산일, 해외 다급 지역 일비, 부장 복지포인트를 각각 알려줘"
#   → 재작성은 한 검색어로 한 문서에 쏠리지만, 분해는 급여·출장·복리 세 문서를 모두 가져온다.
print("decompose() 는 아직 비어 있다 — 위 의사코드를 보고 직접 구현해 보자.")

## STAGE 5 — +하이브리드 검색

**문제.** 의미 검색(dense)은 "연차 ≈ 유급휴가" 같은 뜻은 알아도, `SEC-VPN-01` 이나 사번·코드처럼 글자 그대로 맞춰야 하는 키워드는 자주 놓친다. 반대로 키워드 검색(**BM25**)은 글자는 정확히 잡지만 동의어에 약하다.

**개선.** 둘을 함께 쓰고, 각자 매긴 순위를 **RRF**(Reciprocal Rank Fusion)로 합친다. 이 작고 깨끗한 코퍼스에선 답이 바뀌진 않으니 **검색 순위**로 효과를 본다.

비교: 직전 v4 → 이번 v5.

In [ ]:
# BM25 는 rank-bm25, 한국어 토큰화는 형태소 분석기 kiwipiepy 를 쓴다 (위 '라이브러리 설치' 셀에서 깔았다).
from rank_bm25 import BM25Okapi
from kiwipiepy import Kiwi

_kiwi = Kiwi()


def tokenize(text):
    """형태소 단위로 쪼갠다. '연차를' → '연차'+'를' 처럼 조사를 떼어 BM25 매칭을 높인다 (소문자화로 영문 코드도 매칭)."""
    return [token.form.lower() for token in _kiwi.tokenize(text)]


def reciprocal_rank_fusion(rankings, k=60):
    """여러 검색 결과의 순위를 하나의 점수로 합친다 (RRF)."""
    scores = {}
    for ranking in rankings:
        for rank, index in enumerate(ranking):
            scores[index] = scores.get(index, 0) + 1 / (k + rank)  # 상위(rank 작음)일수록 큰 점수
    return sorted(scores, key=scores.get, reverse=True)


# 하이브리드 검색용 코퍼스 — col_v3 에서 직접 가져온다.
# (search(col_v3) 가 돌려주는 텍스트와 항상 일치하므로 text_to_index 조회에서 KeyError 가 안 난다)
_stored = col_v3.get(include=["documents", "metadatas"])
adv_texts = _stored["documents"]
adv_sources = [m["source"] for m in _stored["metadatas"]]
text_to_index = {text: i for i, text in enumerate(adv_texts)}
bm25 = BM25Okapi([tokenize(text) for text in adv_texts])   # 청크들을 토큰화해 BM25 인덱스 구성


def hybrid(query, k=3):
    """dense(의미) 순위와 BM25(키워드) 순위를 RRF로 결합한다."""
    dense_ranking = [text_to_index[hit["text"]] for hit in search(col_v3, query, k=10)]

    bm25_scores = bm25.get_scores(tokenize(query))
    sparse_ranking = sorted(range(len(adv_texts)), key=lambda i: bm25_scores[i], reverse=True)[:10]

    fused = reciprocal_rank_fusion([dense_ranking, sparse_ranking])[:k]
    return [{"text": adv_texts[i], "source": adv_sources[i]} for i in fused]


def retrieve_hybrid(question, k=4):
    """쿼리 재작성 + 하이브리드 검색."""
    return hybrid(rewrite(question), k=k)


def rag_v5(question):
    """STAGE 5: + 하이브리드."""
    return answer_plain(question, retrieve_hybrid(question))


# code_query 를 바꿔 실험해 보자. 정확히 맞춰야 하는 추천 코드:
#   - "SEC-NB-##"  (외부 반출 노트북)
#   - "SEC-USB-##" (보안 USB)
code_query = "SEC-VPN-01"


def top3_labels(indices):
    """청크 인덱스 목록 → '출처:맥락 첫 줄' 라벨 리스트."""
    labels = []
    for i in indices[:3]:
        first_line = adv_texts[i].splitlines()[0][:18]   # Contextual 로 붙인 맥락 문장(첫 줄)
        labels.append(f"{adv_sources[i]}:{first_line}")
    return labels


dense_top = [text_to_index[hit["text"]] for hit in search(col_v3, code_query, k=3)]
bm25_scores = bm25.get_scores(tokenize(code_query))
bm25_top = sorted(range(len(adv_texts)), key=lambda i: bm25_scores[i], reverse=True)[:3]
hybrid_top = [text_to_index[hit["text"]] for hit in hybrid(code_query, k=3)]

print("dense  상위3:", top3_labels(dense_top))
print("BM25   상위3:", top3_labels(bm25_top))
print("hybrid 상위3:", top3_labels(hybrid_top))
print("\n→ dense 와 BM25 가 올리는 청크 순서가 다르고, hybrid 가 RRF 로 둘을 합친다.")
print("(코드 많고 설명 드문 실제 문서일수록 BM25 기여가 커진다)")

## STAGE 6 — +리랭킹

**문제.** 벡터 검색은 빠르지만 정확도가 떨어져, 넓게 가져올수록 관련 없는 청크도 섞인다.

**개선.** 후보를 넉넉히 뽑은 뒤, 질문과 청크를 한 쌍으로 함께 읽어 관련도를 매기는 **cross-encoder**(`BAAI/bge-reranker-v2-m3`)로 다시 채점해 상위만 남긴다. 답은 그대로지만 **후보가 점수로 어떻게 재정렬되는지** 본다.

비교: 직전 v5 → 이번 v6.

In [ ]:
# 리랭킹은 진짜 cross-encoder(sentence-transformers)를 쓴다.
# (처음 실행 시 BAAI/bge-reranker-v2-m3 모델을 내려받는다 — 수백 MB, 한 번만)
from sentence_transformers import CrossEncoder

reranker = CrossEncoder("BAAI/bge-reranker-v2-m3")


def rerank(question, chunks, top_k=3):
    """cross-encoder 로 (질문, 청크)를 함께 읽어 관련도를 매기고 상위 top_k 만 남긴다."""
    scores = reranker.predict([(question, c["text"]) for c in chunks])
    ranked = sorted(zip(scores, chunks), key=lambda pair: pair[0], reverse=True)
    return [chunk for _, chunk in ranked[:top_k]]


def retrieve_rerank(question, k=4):
    """쿼리 재작성 + 하이브리드 검색으로 후보를 넉넉히 뽑고 리랭킹으로 추린다."""
    candidates = hybrid(rewrite(question), k=8)
    return rerank(question, candidates, top_k=k)


def rag_v6(question):
    """STAGE 6: + 리랭킹."""
    return answer_plain(question, retrieve_rerank(question))


# question 을 바꿔 실험해 보자. 후보에 잡음이 섞이기 쉬운 추천 질문:
#   - "연차 가산일 알려줘"
#   - "비밀번호 규칙이 뭐야?"
# (여기선 흐름을 보려고 1차 검색 후보를 바로 리랭킹한다. 실제 파이프라인은 hybrid 후보를 리랭킹한다.)
question = "출장 숙박비 상한 얼마야?"
candidates = search(col_v3, question, k=6)
print("─ 리랭킹 전 후보 6개 (출처) ─")
print("  ", [c["source"] for c in candidates])

print("\n─ 리랭킹 후 상위 3 (cross-encoder 점수 순) ─")
scores = reranker.predict([(question, c["text"]) for c in candidates])
for score, chunk in sorted(zip(scores, candidates), key=lambda pair: pair[0], reverse=True)[:3]:
    snippet = chunk["text"].splitlines()[-1][:42].strip()
    print(f"   {score:.2f} [{chunk['source']}] {snippet} …")

## STAGE 7 — +출처 인용 (= 고도화 RAG 완성)

**문제.** 검색이 잘돼도, 답이 어느 문서에서 나왔는지 알 수 없거나 규정에 없는 내용을 지어내는 **할루시네이션**이 생길 수 있다.

**개선.** 청크마다 번호를 매겨 넣고, 답의 각 문장 끝에 근거 번호 `[n]` 을 달게 한다. 이걸로 모든 STAGE를 쌓았다 — `rag_advanced` 가 곧 **고도화 RAG**다.

비교: 직전 v6 → 이번 v7.

In [ ]:
def answer_cited(question, chunks):
    """청크에 번호를 붙여 넣고, 각 문장 끝에 근거 번호 [n] 을 달게 한다."""
    context = "\n\n".join(f"[{i + 1}] ({c['source']}) {c['text']}" for i, c in enumerate(chunks))
    response = client.chat.completions.create(
        model=CHAT_MODEL,
        temperature=0,
        messages=[
            {"role": "system", "content":
                "사내 문서를 근거로만 답하고, 없으면 모른다고 하라. "
                "각 문장 끝에 근거 번호를 [1]처럼 달아라."},
            {"role": "user", "content": f"[문서]\n{context}\n\n[질문]\n{question}"},
        ],
    )
    return response.choices[0].message.content


def rag_advanced(question):
    """STAGE 7: + 출처 인용. 모든 개선을 쌓은 고도화 RAG."""
    return answer_cited(question, retrieve_rerank(question))


# question 을 바꿔 실험해 보자. 답은 같아도 [n] 근거가 붙는지 본다:
#   - "출장 숙박비 상한은 직급별로 얼마야?"
question = "본인이 결혼하면 경조금은 얼마야?"
compare_stage(question, rag_v6, rag_advanced, "v6 인용 없음", "v7 +출처 인용")

### STAGE 0 → 7 한눈에 — 기본 RAG vs 고도화 RAG

여덟 단계를 다 쌓았다. 구어체로 던진 표 질문을 **맨 처음 기본 RAG**와 **완성된 고도화 RAG**에 각각 넣어 차이를 본다.

In [ ]:
# question 을 바꿔 실험해 보자. 기본은 막히고 고도화는 푸는 추천 질문 (구어체 + 표 속 값):
#   - "노트북 집에 가져가도 돼?"
#   - "부장의 연봉 환산 금액은 얼마야?"
question = "내가 부장이면 연봉이 얼마로 잡혀?"
compare_stage(question, rag_v0, rag_advanced, "STAGE 0 기본 RAG", "STAGE 7 고도화 RAG")

# 더 챙길 것 — 에이전트·보안·평가

누적 단계 위에, 실무에서 빠뜨리기 쉬운 세 가지를 덧붙인다.

## 에이전트형 RAG

**문제.** 지금까진 "검색 → 생성"이 한 방향으로 한 번 흘렀다. 그런데 서로 다른 문서에 흩어진 복합 질문은 한 번의 검색으로 한쪽만 답하기 쉽다.

**개선.** **에이전트형 RAG**는 검색을 한 번에 끝내지 않는다. 결과를 보고 "충분한가, 더 찾아야 하나?"를 스스로 판단하며 반복한다(검색 단계가 화면에 찍힌다).

In [ ]:
def agentic(question, max_steps=3):
    """하이브리드로 검색 → '충분한가?' 판단 → 부족하면 키워드 바꿔 재검색, 충분하면 답한다."""
    gathered = []
    seen = set()
    sub_query = question

    for step in range(max_steps):
        for hit in hybrid(sub_query, k=3):
            if hit["text"] not in seen:
                seen.add(hit["text"])
                gathered.append(hit)

        context = "\n\n".join(f"({h['source']}) {h['text']}" for h in gathered)
        response = client.chat.completions.create(
            model=CHAT_MODEL,
            temperature=0,
            messages=[{"role": "user", "content":
                f"질문:{question}\n모은 문서:\n{context}\n\n"
                "질문의 모든 부분을 문서로 확인했을 때만 'ANSWER: <답>'. "
                "아직 근거를 못 찾은 부분이 있으면 그 부분만 'SEARCH: <키워드>' 로. 한 줄만 출력."}],
        )
        line = response.choices[0].message.content.strip()

        if line.startswith("ANSWER:"):
            return line[len("ANSWER:"):].strip()

        sub_query = line.split("SEARCH:", 1)[-1].strip()
        print(f"  step{step} 추가 검색: {sub_query}")

    return answer_cited(question, gathered)


# question 을 바꿔 실험해 보자. 서로 다른 문서를 거쳐야 하는 멀티홉 추천 질문:
#   - "출장 숙박비 상한은 얼마이고, 비밀번호는 며칠마다 바꿔?"
question = "출산하면 받는 축하금은 얼마이고, 출산휴가는 며칠이야?"
print("[기본 RAG]\n" + rag_v0(question))
print("\n[에이전트형 RAG]")
print(agentic(question))

## 보안·권한

취업규칙·급여 같은 **민감한 사내 문서**에선, 일반 직원이 "다른 사람 연봉?"을 물었을 때 급여 문서가 검색돼선 안 된다. 인덱싱 때 청크에 권한 태그를 달고, 검색 단계에서 **사용자 권한으로 메타데이터 필터링**해 볼 수 있는 문서만 후보로 둔다.

> 프롬프트 인젝션 대비: 검색해 온 내용은 명령이 아니라 참고 자료로만 다룬다.

In [ ]:
# 문서별 열람 권한 (예: 급여규정은 'hr' 권한이 있어야 본다)
DOC_ACL = {"급여규정": "hr"}


def answer_with_acl(question, user_roles, k=3):
    """사용자 권한으로 문서를 거르고, 권한이 없어 답할 수 없으면 그렇게 안내한다."""
    # 1) 권한이 필요한 문서가 가장 관련 높은데 사용자에게 권한이 없으면 → 차단 안내
    top = search(col_v3, question, k=1)[0]
    needed = DOC_ACL.get(top["source"])
    if needed and needed not in user_roles:
        return f"'{top['source']}' 는 열람 권한({needed})이 필요합니다 — 권한이 없어 답해 드릴 수 없습니다."
    # 2) 볼 수 있는 문서만 후보로 검색해 답한다
    blocked = [src for src, role in DOC_ACL.items() if role not in user_roles]
    where = {"source": {"$nin": blocked}} if blocked else None
    return answer_cited(question, search(col_v3, question, k=k, where=where))


question = "과장 월 기본급은 얼마야?"

print("─ 일반 직원 (권한 없음) ─")
print("  답:", answer_with_acl(question, user_roles=[]))

print("\n─ 인사팀 (hr 권한 있음) ─")
print("  답:", answer_with_acl(question, user_roles=["hr"]))

## 평가 — 기본 vs 고도화를 숫자로

"정말 좋아졌는지는 측정해 봐야 안다." 정답 라벨 없이 강한 LLM 에게 채점을 맡기는 **LLM-as-Judge** 가 표준이다(실무: RAGAS·DeepEval·TruLens).
같은 질문에 STAGE 0 검색과 고도화 검색을 충실성·관련성으로 비교한다.

In [ ]:
def judge(question, answer, chunks):
    """답이 문서에 근거하나(충실성)·질문에 답하나(관련성)를 각각 0~1로 채점한다."""
    context = "\n\n".join(c["text"] for c in chunks)
    response = client.chat.completions.create(
        model=CHAT_MODEL,
        temperature=0,
        messages=[{"role": "user", "content":
            f"질문:{question}\n문서:{context}\n답변:{answer}\n\n"
            "충실성(문서 근거)·관련성(질문에 답)을 각각 0~1로. 형식: 충실성=<수>, 관련성=<수>"}],
    )
    return response.choices[0].message.content.strip()


question = "부장의 연봉 환산 금액은 얼마야?"
basic_hits = search(col_v0, question, k=3)
advanced_hits = retrieve_rerank(question)
basic_answer = answer_plain(question, basic_hits)
advanced_answer = answer_cited(question, advanced_hits)

print("질문:", question)
print("\n[STAGE 0 기본] 답:", basic_answer)
print("  평가:", judge(question, basic_answer, basic_hits))
print("\n[고도화] 답:", advanced_answer)
print("  평가:", judge(question, advanced_answer, advanced_hits))

**새 모델이 나오면?** 더 싸고 좋다는 임베딩이 나와도 우리 문서에서 더 좋은지는 보장할 수 없다 —
이 평가셋을 한 번 만들어 두고 바꿀 때마다 점수를 비교해 채택/롤백한다(소프트웨어 **회귀 테스트**).

> **평가셋 만들기:** 실제 직원 질문 로그에서 대표 질문을 뽑거나, 각 문서에서 LLM으로 "질문–정답 근거" 쌍을 생성해 사람이 검수한다. 표·다중문서 같은 까다로운 유형을 섞는다.

### RAGAS 로도 평가하기 (실무 표준)

위 `judge` 는 개념을 보여 주는 최소 구현이다. 실무에선 **[RAGAS](https://docs.ragas.io/)** 같은 전용 라이브러리가 충실성·관련성·문맥 정밀도/재현율을 표준화된 방식으로 채점한다. 같은 답을 RAGAS 로 평가해 본다.

> ragas 는 옛 langchain 과 맞물려 최신 버전과 충돌한다. 그래서 호환 버전(`langchain<1.0`)을 함께 설치한다 — 아래 셀이 처음 실행될 때 자동으로 맞춰 깐다.

In [ ]:
# RAGAS 가 설치돼 있으면 평가한다 (선택). 없으면 안내만 하고 건너뛴다.
try:
    from ragas import evaluate, EvaluationDataset
    from ragas.dataset_schema import SingleTurnSample
    from ragas.metrics import Faithfulness, ResponseRelevancy
    from ragas.llms import LangchainLLMWrapper
    from ragas.embeddings import LangchainEmbeddingsWrapper
    from langchain_openai import ChatOpenAI, OpenAIEmbeddings
    RAGAS_AVAILABLE = True
except ImportError as e:
    RAGAS_AVAILABLE = False
    print("⚠️ RAGAS 미설치 — 건너뜁니다:", e)
    print("   설치: pip install 'ragas==0.2.15' 'langchain<1.0' 'langchain-community<0.4' langchain-google-vertexai langchain-openai")

if RAGAS_AVAILABLE:
    # question 을 바꿔 평가해 보자.
    question = "대리 연차 가산일은 며칠이고, 해외 다급 지역 일비는 얼마야?"
    chunks = retrieve_rerank(question)                  # 고도화 RAG 의 검색 결과
    sample = SingleTurnSample(
        user_input=question,
        response=answer_cited(question, chunks),
        retrieved_contexts=[c["text"] for c in chunks],
    )
    judge_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4o-mini", temperature=0))
    judge_emb = LangchainEmbeddingsWrapper(OpenAIEmbeddings(model="text-embedding-3-small"))

    result = evaluate(
        EvaluationDataset(samples=[sample]),
        metrics=[Faithfulness(llm=judge_llm), ResponseRelevancy(llm=judge_llm, embeddings=judge_emb)],
    )
    scores = result.to_pandas()
    print("질문 :", question)
    print("답변 :", sample.response.replace("\n", " "))
    print(f"\n  충실성 faithfulness     : {scores['faithfulness'][0]:.3f}   (답이 근거 문서에 있나)")
    print(f"  관련성 answer_relevancy : {scores['answer_relevancy'][0]:.3f}   (답이 질문에 답하나)")

## 정리 — 단계별로 무엇이 달라졌나

| STAGE | 적용한 개선 | 무엇이 좋아졌나 | 비용 |
|---|---|---|---|
| 1 | 표보존 추출 | 표의 행·열이 살아남(데이터) | 레이아웃 파서 호출 |
| 2 | 구조 청킹 | 표·항목이 한 청크에 → 표 질문에 답 | 거의 없음 |
| 3 | Contextual | 막연한 질문도 맞는 청크 매칭 | 청크마다 LLM |
| 4 | 쿼리 재작성 | 구어체·어긋난 질문도 맞는 문서로 | 질문마다 LLM 한 번 |
| 5 | 하이브리드 | 코드·키워드까지 안정적으로 | BM25·토큰화 관리 |
| 6 | 리랭킹 | 후보 잡음 제거, 정밀도↑ | 후보 재채점 느림 |
| 7 | 출처 인용 | 근거 확인·환각↓ | 거의 없음 |

전부 넣을 필요는 없다. **평가로 병목을 찾고 그 단계부터** 붙이면 된다.
개념 설명은 블로그 글 [RAG 고도화 및 최적화](https://seojuny.dev/advanced-rag) 참고.

# 마무리 — 프레임워크(LangChain)로는 이렇게 짧다

여기까지 RAG 파이프라인을 함수 하나하나 직접 만들어 봤다. 사실 실무에선 **[LangChain](https://www.langchain.com/)** 같은 프레임워크를 쓰면 같은 파이프라인을 몇 줄로 끝낼 수 있다. 적재·청킹·임베딩·검색·생성이 이미 컴포넌트로 갖춰져 있어, 가져다 연결하기만 하면 된다.

그렇다면 처음부터 프레임워크를 쓰면 되지 않았을까? 그렇지 않다. 프레임워크는 내부를 가려 줘서 빠르게 만드는 대신, **안에서 무슨 일이 일어나는지 모르면 디버깅도 개선도 어렵다.** 검색이 관련 없는 걸 가져올 때 청킹이 문제인지 임베딩이 문제인지, 표가 왜 안 잡히는지, 어디에 하이브리드나 리랭킹을 끼워야 하는지 — 원리를 알아야 짚어낸다. 우리가 단계마다 직접 뜯어 본 게 바로 그 원리다.

아래는 위에서 단계별로 쌓은 **고도화 RAG 전체**를 LangChain 컴포넌트로 똑같이 조립한 코드다. 우리가 직접 구현한 게 프레임워크엔 부품으로 들어 있다 — 구조 청킹(`RecursiveCharacterTextSplitter`)·하이브리드(`EnsembleRetriever`+`BM25Retriever`)·쿼리 변환(`MultiQueryRetriever`)·리랭킹(`CrossEncoderReranker`)·출처 인용까지. **직접 돌려 보자**(처음 실행 시 자동 설치). 무슨 부품이 어디서 무얼 하는지는, 위에서 직접 만들어 봤기에 이제 안다.

In [ ]:
# 고도화 RAG 를 LangChain 으로 조립한다 (선택). 라이브러리가 있으면 실행, 없으면 건너뛴다.
# (langchain 은 0.3 으로 고정 — 위 '라이브러리 설치' 셀에서 맞췄다.)

# 직접 돌려 볼 질문 — 바꿔도 된다.
question = "부장 연봉 환산은 얼마이고, 본인 결혼 경조금은 얼마야?"

try:
    from langchain.retrievers import EnsembleRetriever, ContextualCompressionRetriever
    from langchain.retrievers.document_compressors import CrossEncoderReranker
    from langchain.retrievers.multi_query import MultiQueryRetriever
    from langchain_community.document_loaders import PyMuPDFLoader
    from langchain_community.retrievers import BM25Retriever
    from langchain_community.cross_encoders import HuggingFaceCrossEncoder
    from langchain_text_splitters import RecursiveCharacterTextSplitter
    from langchain_openai import OpenAIEmbeddings, ChatOpenAI
    from langchain_chroma import Chroma
    from langchain_core.prompts import ChatPromptTemplate
    from langchain_core.runnables import RunnablePassthrough
    from langchain_core.output_parsers import StrOutputParser
    LANGCHAIN_AVAILABLE = True
except ImportError as e:
    LANGCHAIN_AVAILABLE = False
    print("⚠️ LangChain 미설치 — 건너뜁니다:", e)
    print("   설치: pip install 'langchain-core<1.0' 'langchain<1.0' 'langchain-community<0.4' 'langchain-openai<1.0' 'langchain-chroma<1.0' 'langchain-text-splitters<1.0' rank-bm25 sentence-transformers")

if LANGCHAIN_AVAILABLE:
    from pathlib import Path
    llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

    # STAGE 1 표보존: PyMuPDFLoader 로 적재 (표를 더 살리려면 pymupdf4llm 으로 Markdown 변환)
    docs = []
    for pdf in Path("../data_advanced").glob("*.pdf"):
        docs += PyMuPDFLoader(str(pdf)).load()

    # STAGE 2 구조 청킹: 제N조 / ## 헤딩 경계부터 우선해 자른다
    chunks = RecursiveCharacterTextSplitter(
        separators=["\n## ", "\n제", "\n\n", "\n", " "], chunk_size=700, chunk_overlap=60
    ).split_documents(docs)

    store = Chroma.from_documents(chunks, OpenAIEmbeddings(model="text-embedding-3-small"))
    dense = store.as_retriever(search_kwargs={"k": 10})

    # STAGE 5 하이브리드: dense + BM25(위 STAGE 5의 kiwi 토크나이저 재사용)를 RRF로 묶는다
    bm25 = BM25Retriever.from_documents(chunks, preprocess_func=tokenize)
    bm25.k = 10
    hybrid_retriever = EnsembleRetriever(retrievers=[dense, bm25], weights=[0.5, 0.5])

    # STAGE 4 쿼리 변환: 질문을 여러 버전으로 만들어 각각 검색한다 (멀티쿼리)
    multi_query = MultiQueryRetriever.from_llm(retriever=hybrid_retriever, llm=llm)

    # STAGE 6 리랭킹: cross-encoder(bge)로 후보를 다시 채점해 상위 4개만 남긴다
    reranker = CrossEncoderReranker(
        model=HuggingFaceCrossEncoder(model_name="BAAI/bge-reranker-v2-m3"), top_n=4
    )
    retriever = ContextualCompressionRetriever(base_compressor=reranker, base_retriever=multi_query)

    # STAGE 7 출처 인용: 각 문장 끝에 근거 번호를 달게 한다
    def format_docs(docs):
        return "\n\n".join(f"[{i+1}] ({d.metadata.get('source', '')}) {d.page_content}" for i, d in enumerate(docs))

    prompt = ChatPromptTemplate.from_template(
        "사내 문서를 근거로만 답하고, 없으면 모른다고 하라. 각 문장 끝에 근거 번호를 [1]처럼 달아라."
        "\n\n[문서]\n{context}\n\n[질문]\n{question}"
    )
    chain = (
        {"context": retriever | format_docs, "question": RunnablePassthrough()}
        | prompt
        | llm
        | StrOutputParser()
    )
    print(chain.invoke(question))

    # STAGE 3 Contextual 은 스톡 컴포넌트가 없어, 인덱싱 때 청크에 맥락 한 문장을 붙이는
    # 커스텀 단계로 더한다 (위 add_context 와 같은 아이디어).

프레임워크는 **원리를 이해한 다음** 생산성을 위해 쓰는 도구다. 순서가 바뀌면, 문제가 생겼을 때 손쓸 수가 없다.